In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar `creditsight` sin instalarlo: el código vive en ejemplo/src
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))


# 01 · Caso de negocio — CreditSight

Antes de abrir un solo dato, el proyecto necesita **una pregunta de negocio, una métrica que la
refleje y un costo de equivocarse**. Esta etapa produce la *ficha de caso* (`reports/00_caso.md`),
que es lo primero que el instructor aprueba en tu grupo.

## La pregunta

*CreditSight* es una fintech que ofrece líneas de crédito digitales. Para cada cliente debe decidir
si le asigna **un límite alto, uno estándar o uno restringido**. Hoy el analista de riesgo revisa a
mano el historial y tarda días.

> **¿Podemos clasificar a cada cliente como `Good`, `Standard` o `Poor` a partir de su perfil
> financiero y su historial de pagos, para priorizar la revisión del analista y acelerar las
> aprobaciones?**

La persona que usa el resultado es el **analista de riesgo**, que conserva la decisión final. El
modelo no decide solo: ordena la cola de trabajo.

## Qué tiene de "avanzado" este caso

| Decisión | Por qué importa |
|---|---|
| Unidad de análisis: **cliente-mes** | Cada cliente aparece 8 veces. Cómo se parte el dato decide si la métrica es honesta |
| Errores con **costo asimétrico** | Aprobar a un cliente `Poor` cuesta mucho más que rechazar a uno `Good` |
| **Datos sucios** reales | Valores corruptos que hay que entender antes de limpiar |
| **Datos personales** | Nombre y SSN no pueden llegar al modelo |

## La matriz de costos

Equivocarse no cuesta lo mismo en las dos direcciones. Convertimos esa idea en números con una
**matriz de costos**: filas = clase real, columnas = clase predicha. Son unidades de costo relativas,
no dólares; lo importante es la proporción (aprobar a un `Poor` como `Good` cuesta 10, rechazar a un
`Good` como `Poor` cuesta 2).

In [2]:
import pandas as pd

from creditsight.config import CLASSES, COST_MATRIX

cost = pd.DataFrame(COST_MATRIX, index=[f"real {c}" for c in CLASSES], columns=[f"predice {c}" for c in CLASSES])
cost

,predice Poor,predice Standard,predice Good
real Poor,0.0,3.0,10.0
real Standard,1.0,0.0,3.0
real Good,2.0,1.0,0.0


## ¿Contra qué nos comparamos?

Una métrica sola no dice nada: necesita un punto de referencia. Con las proporciones de cada clase
(53,2 % `Standard`, 29,0 % `Poor`, 17,8 % `Good`, que el EDA confirmará) calculamos el **costo
esperado por cliente** de dos estrategias sin modelo:

- **Siempre `Standard`** (la clase más frecuente).
- **Al azar** según las proporciones de cada clase.

Todo modelo que construyamos tiene que ganarles, y por un margen que justifique su costo.

In [3]:
import numpy as np

priors = np.array([0.290, 0.532, 0.178])  # Poor, Standard, Good (se confirman en el EDA)

always_standard = priors @ COST_MATRIX[:, 1]
random_by_priors = priors @ COST_MATRIX @ priors
print(f"Siempre 'Standard':        costo esperado por cliente = {always_standard:.2f}")
print(f"Al azar según proporciones: costo esperado por cliente = {random_by_priors:.2f}")

Siempre 'Standard':        costo esperado por cliente = 1.05
Al azar según proporciones: costo esperado por cliente = 1.62


## Cómo se medirá el éxito

- **Métrica principal:** costo esperado por cliente-mes con la matriz anterior (menor es mejor).
- **Controles:** F1 macro (ninguna clase puede ignorarse) y recall de la clase `Poor` (la que más
  cuesta no detectar).
- **No se usa exactitud (accuracy):** la clase mayoritaria sola ya da ~53 %.
- **Regla de decisión:** no se elige la clase más probable sino la de **menor costo esperado**.

## Riesgos que el caso debe reconocer desde el inicio

- **Equidad:** el modelo no debe usar atributos protegidos ni sus sustitutos obvios sin analizarlos;
  se revisará el error por segmento en la etapa de validación.
- **Datos personales:** `Name`, `SSN` e `ID` se eliminan al cargar los datos.
- **Uso responsable:** el resultado apoya al analista, no reemplaza una decisión de crédito.

## Entregable de esta etapa

La ficha de caso está en `ejemplo/reports/00_caso.md`. En tu proyecto, la tuya sigue el formato de
`curso/evaluacion/ficha-de-caso.md`.